# Ten Trails — USG in, MODFLOW 6 out

Two models, side by side:

| | what it is | how you drive it |
|---|---|---|
| **`usg`** | the MODFLOW-USG model, read | `usg.k`, `usg.botm`, `usg.boundaries[...]`, `usg.cln`, plus its own 2023 run in `tr-ex1-5.hds` |
| **`model`** | the converted MODFLOW 6 model, run | myflopy's read grammar: `model.packages.<pkg>.<inputs\|results>.<noun>.<verb>()`, `model.plot`, `model.budget` |

**Where this stands (2026-08-29).** The conversion is complete and tested. The full
converted model **solves stress period 1 and fails to converge at period 2 of 72**.
Bisected: `CHD + GHB + DRN` alone reaches normal termination, so the grid, properties and
list boundary conditions are sound — the instability arrives with `RCH` and/or `EVT`.

So §6 builds a **short, converging** run you can actually explore, and §9 is set up to
finish the bisect on the full one. Leading hypothesis, *not yet proven*: recharge reaches
2.08 ft/day locally (typical ≈0.009). That is genuinely in the file — the multipliers are
1/(12·31), so raw units are inches/month, and this is a stormwater-infiltration model. In
USG that water drained into the CLN network, which MODFLOW 6 has no package for.


## 1 · Setup

In [1]:
import re, shutil, subprocess, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import flopy

import myflopy as mf
import myflopy.package_api as api
from myflopy.specs import ModelSpec, SimulationSpec

warnings.filterwarnings('ignore')

# NOTE there are two name files. They differ ONLY in DISU and OC: flow-tt01_USE.nam runs
# 312 periods, flow-tt01_USE_5yr.nam runs 72. d_model.bat runs the _5yr one and
# tr-ex1-5.hds is its output, so that is the one to read.
USG_DIR = Path('/home/lukem/models/mf6/Ten Trails/SSPA/Model_FINAL_ForLuke/14-Calib40_Iter01_DV')
NAM = USG_DIR / 'flow-tt01_USE_5yr.nam'
GSF = USG_DIR / 'flow-tt01.gsf'
CRS = 'EPSG:2927'                       # WA State Plane South, US survey feet
START = '2017-10-01'                    # the DISU date column is unusable -- see the log

WORK = Path('/tmp/usg_workbook'); WORK.mkdir(exist_ok=True)
MF6 = shutil.which('mf6') or '/home/lukem/.local/bin/mf6'


def show(pic, title=None):
    """Show a Picture, optionally titled.

    `plot.map(..., title=...)` does NOT work: unrecognised keywords go to the Plotly
    *trace*, and a choropleth trace has no `title`, so it raises when the figure is
    built. Titles belong on the layout.
    """
    if title:
        pic.fig.update_layout(title=title, margin=dict(t=44))
    pic.show()
    return pic

print('mf6:', MF6)


mf6: /home/lukem/.local/bin/mf6


The reader talks through `logging`, not `print`. Turn it on — it says when it refuses to guess.

In [2]:
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)-7s %(message)s', force=True)
logging.getLogger('myflopy').setLevel(logging.INFO)   # DEBUG for per-array detail


# Part A — the imported USG model

## 2 · Read it

In [3]:
usg = mf.read_usg(NAM, gsf=GSF, crs=CRS)
usg


INFO    reading MODFLOW-USG model flow-tt01_USE_5yr.nam (BAS6, SMS, DISU, OC, RCH, WEL, DRN, CHD, LPF, CLN, GHB, ETS, HFB6)
INFO    building the Voronoi grid
INFO    Voronoi grid ready: 9405 cells
WARNING flow-tt01_5yr.dis carries a date on each stress-period line, but the dates do not increase (2023-10-31 then 2023-01-31) -- so they cannot give a start date. Pass start_date_time= to to_mf6() to set one.
INFO    read 5 layers x 9405 cells, 72 periods, 4 boundary package(s), CLN with 804 nodes


UsgModel('flow-tt01_USE_5yr.nam', 5 layers x 9405 cells, 72 periods, packages=['CHD', 'DRN', 'GHB', 'WEL'])

`report()` is the point of the exercise: a conversion may be lossy, never *silently* lossy.
Every approximation and omission is counted here.

In [ ]:
print(usg.report())


`validate()` finds what MODFLOW 6 will **refuse** before you spend a run on it — mostly a
head boundary below the bottom of its own cell, which USG tolerates and MF6 does not.

In [ ]:
for f in usg.validate():
    print(f'[{f.severity:>6}] {f.detail}')
    print(f'          fix: {f.fix}\n')


## 3 · Grid, layers, properties

A USG `DISU` has no coordinates — the geometry lives in the `.gsf`.

In [ ]:
vor = usg.grid
print(f'{usg.nlay} layers x {usg.ncpl} cells = {usg.nodes} nodes, {usg.nper} stress periods')
areas = vor.gdf_vorPolys.area
print(f'cell area {areas.min():,.1f} to {areas.max():,.0f} ft2 (median {areas.median():,.0f})')

# The DISU states cell areas independently of the .gsf polygons -- a free cross-check.
rel = np.abs(areas.to_numpy() - np.asarray(usg.disu.area)[:usg.ncpl]) / np.asarray(usg.disu.area)[:usg.ncpl]
print(f'DISU area vs .gsf polygon area: median {np.median(rel):.2e}, max {rel.max():.2e}')
vor.plot.grid().show()


In [ ]:
pd.DataFrame({
    'botm_mean':  usg.botm.mean(axis=1),
    'thick_min':  usg.thickness.min(axis=1),
    'thick_mean': usg.thickness.mean(axis=1),
    'cells < 1ft':(usg.thickness < 1).sum(axis=1),
    'k_min':      usg.k.min(axis=1),
    'k_max':      usg.k.max(axis=1),
    'k33_min':    usg.k33.min(axis=1),
    'sy_mean':    usg.sy.mean(axis=1),
    'active':     (usg.idomain != 0).sum(axis=1),
}, index=[f'layer {i+1}' for i in range(usg.nlay)]).round(4)


K spans five orders of magnitude, and layer 1 gets down to a tenth of a foot thick — both worth seeing.

In [ ]:
LAYER = 0   # change me
show(vor.plot.map(usg.k[LAYER], logscale=True), f'Kx, layer {LAYER+1} (ft/d)')


In [ ]:
show(vor.plot.map(usg.thickness[LAYER]), f'thickness, layer {LAYER+1} (ft)')


## 4 · Boundary conditions and stresses

In [ ]:
for ft, rec in usg.boundaries.items():
    print(f'{ft:<5} {rec.n_defined:>3} periods defined, {len(rec.reused):>3} reused, '
          f'{rec.n_records:>6} GWF records, {rec.n_cln_records:>6} CLN records')


`DRN`/`GHB` define period 1 and reuse it for the rest — USG's `ITMP < 0`, which MODFLOW 6
says the same way by omitting the period. `WEL` has **zero** groundwater records: all 704
per period are P−ET on CLN nodes. **There is no pumping anywhere in this model.**

Each boundary comes back as a GeoDataFrame with real geometry, so it survives a re-grid.

In [ ]:
drn = usg.boundary_frame('DRN')
chd = usg.boundary_frame('CHD')
ghb = usg.boundary_frame('GHB')
paint = np.full(usg.ncpl, np.nan)
paint[drn['cell'].unique()] = 1
paint[chd['cell'].unique()] = 2
paint[ghb['cell'].unique()] = 3
show(vor.plot.map(paint), '1 = DRN   2 = CHD   3 = GHB')


The CHD is not a constant — 12 stages repeating annually, ~2.6 ft of swing, on the northern margin.

In [ ]:
stage = pd.Series({p: b[:, 1].max() for p, b in sorted(usg.boundaries['CHD'].periods.items())})
print(f'{stage.nunique()} distinct stages, {stage.min():.2f}-{stage.max():.2f} ft')
stage.plot(figsize=(11, 3), marker='o', ms=3, title='CHD stage by stress period (ft)');


### The prime suspect

In [ ]:
R = np.array([usg.rch.rech[p] for p in sorted(usg.rch.rech)])
E = np.array([usg.ets.rate[p]  for p in sorted(usg.ets.rate)])
print(f'RCH {R.shape}  {R.min():.4g} to {R.max():.4g} ft/d  (mean of nonzero {R[R>0].mean():.4g})')
print(f'EVT {E.shape}  {E.min():.4g} to {E.max():.4g} ft/d')
print(f'\n40 in/yr of recharge, for scale = {40/12/365:.5f} ft/d')
print(f'cells over 0.1 ft/d in some period: {(R.max(axis=0) > 0.1).sum()} of {usg.ncpl}')
show(vor.plot.map(R.max(axis=0), logscale=True), 'peak recharge over the run (ft/d)')


ET became a **list** package: `NETSEG = 2`, and MODFLOW 6 cannot combine segments with
`READASARRAYS`. `NETSOP = 3` (ET to the highest *active* cell) has no MF6 equivalent either,
but IBOUND is static so resolving it once is exact.

In [ ]:
print(f'ETS NETSOP={usg.ets.netsop}  NETSEG={usg.ets.netseg}')
print(f'  pxdp={usg.ets.pxdp[0][0][0]:.2f}  petm={usg.ets.petm[0][0][0]:.2f}  '
      f'extinction depth {usg.ets.depth[0].min():.1f}-{usg.ets.depth[0].max():.1f} ft')
print(f'  EVT records written: {usg.has_active_column.sum():,} cols x {usg.nper} per '
      f'= {usg.has_active_column.sum()*usg.nper:,}')


## 5 · What the CLN took with it

MODFLOW 6 has no Connected Linear Network. The 804 nodes are read and segmented by graph
shape — a chain is a stream, a 2-D mesh is a water body — but nothing is written. These
polygons are what a `LAK`/`SFR` rebuild would start from.

In [ ]:
clnp = usg.cln_polygons()
display(clnp[['feature', 'kind', 'n_nodes', 'n_cells', 'elev_min', 'elev_max']])
paint = np.full(usg.ncpl, np.nan)
for f in usg.cln.features:
    paint[f.cells] = f.index
show(vor.plot.map(paint), 'CLN features (0-3 water bodies, 4-5 streams) — NOT converted')


In [ ]:
wel = usg.boundaries['WEL']
q = np.array([wel.cln_periods[p][:, 1] for p in sorted(wel.cln_periods)])
print(f'CLN P-ET: {q.shape[1]} records per period x {q.shape[0]} periods')
print(f'net flux per period: mean {q.sum(axis=1).mean():+,.0f} ft3/d '
      f'(min {q.sum(axis=1).min():+,.0f}, max {q.sum(axis=1).max():+,.0f})')
pd.Series(q.sum(axis=1)).plot(figsize=(11, 3), title='net P-ET onto the CLN water bodies (ft3/d)');


## 6 · The USG model's own results

`tr-ex1-5.hds` is the last real MODFLOW-USG run. It lets you check MF6 without needing the
Windows executable. `999` is USG's HNOFLO — inactive, not a head.

In [ ]:
usg_hds = flopy.utils.HeadUFile(str(USG_DIR / 'tr-ex1-5.hds'))
usg_kk = usg_hds.get_kstpkper()
saved = sorted({k[1] for k in usg_kk})
print(f'{len(usg_kk)} records covering stress periods {saved[0]+1}-{saved[-1]+1} '
      f'({len(saved)} of {usg.nper}) -- its OC saved nothing earlier')

def usg_head(period, layer=0):
    """USG heads for one stress period, HNOFLO masked out."""
    k = max(kk for kk in usg_kk if kk[1] == period)
    a = np.array(usg_hds.get_data(kstpkper=k))
    return np.where(np.isclose(a, 999.0), np.nan, a)[layer]

show(vor.plot.map(usg_head(usg_kk[-1][1]), contours=True),
     f'USG head, layer 1, period {usg_kk[-1][1]+1} (ft)')


# Part B — the converted MODFLOW 6 model

## 7 · Convert and run

Building it as a **`Project` run** (rather than just writing files) is what gives you
myflopy's read grammar afterwards: `model.packages.<pkg>.<inputs|results>.<noun>.<verb>()`.

> **Never set `local_origin=False`.** MODFLOW 6 builds DISV conductances from raw vertex
> coordinates; on State Plane (~1.34 M ft) it loses the precision and returns a **NaN budget
> while printing "Normal termination"**. Measured: 0 of 9,405 cells finite as-is, 9,405 of
> 9,405 shifted. The default keeps the model georeferenced via `xorigin`/`yorigin`.

In [4]:
def truncate(spec, nper):
    """Limit every package's period data to the first `nper` periods.

    Without this a 4-period test still writes all 72 periods of EVT -- 61 MB and
    minutes of waiting -- and MF6 warns about every period past nper.
    """
    keys = ('stress_period_data', 'recharge', 'irch', 'transient', 'steady_state')
    packages = []
    for pkg in spec.models[0].packages:
        cut = {k: {i: v for i, v in pkg.options[k].items() if i < nper}
               for k in keys if isinstance(pkg.options.get(k), dict)}
        packages.append(pkg.with_options(**cut) if cut else pkg)
    model = ModelSpec(spec.models[0].name, 'gwf', packages=tuple(packages),
                      context=spec.models[0].context, options=spec.models[0].options)
    sim_pkgs = [q for q in spec.packages if q.name != 'tdis']
    sim_pkgs.append(api.tdis(nper=nper, perioddata=[(31.0, 1, 1.1)] * nper, time_units='days'))
    return SimulationSpec(spec.name, models=(model,), packages=tuple(sim_pkgs))


def build_run(name, *, include=None, nper=None, root=WORK / 'project'):
    """Convert, register as a Project run, execute, and hand back the Run."""
    spec = usg.to_mf6('tentrails', crs=CRS, start_date_time=START,
                      fix_for_mf6=True, include=include)
    if nper:
        spec = truncate(spec, nper)
    project = mf.Project(root, name='tentrails')
    project.add_simulation(spec)
    run = project.prepare_run(name, spec)
    ok, _ = run.execute()
    print(f'{name}: execute -> {ok}   workspace {run.workspace}')
    return run


### A run that finishes

The full model stops at period 2, so start with the subset that reaches normal termination.
This is the MF6 model to explore; §9 diagnoses the full one.

In [5]:
shutil.rmtree(WORK / 'project', ignore_errors=True)
run = build_run('short', include=('chd', 'ghb', 'drn'), nper=12)
model = run.model('tentrails')
model


WARNING CHD: omitted 348 record-period(s) whose head sits below the cell bottom -- inert in MODFLOW-USG, rejected by MODFLOW 6 (fix_for_mf6=True)
WARNING GHB: raised 6 boundary head(s) to their cell bottom so MODFLOW 6 accepts them (fix_for_mf6=True)
INFO    converted flow-tt01_USE_5yr.nam to MODFLOW 6: 8 packages on a 5 x 9405 DISV grid, 72 periods


short: execute -> False   workspace /tmp/usg_workbook/project/runs/short


## 8 · Interrogating the MF6 model

Now the myflopy grammar applies. Every noun answers the same verbs —
`get` / `summary` / `plot`, plus `map` / `section` / `mosaic` / `animate` for spatial ones.

In [ ]:
model.file_summary()


In [ ]:
model.grid_summary()


### Budget — where the water goes

In [ ]:
model.budget_incremental.head(12)   # a property, not a method


In [ ]:
bud = model.budget_incremental.set_index('stress_period').drop(columns=['totim', 'time_step'])
bud.plot(figsize=(12, 4), title='incremental budget by stress period');


### Heads

In [ ]:
# `model.hds` is a HeadsPlus object (a property); `.get()` gives the DataFrame.
print('head records:', model.hds.get_nrecords(), ' periods:', len(model.hds.get_kstpkper()))
model.plot.map(layer=2, per=5).show()          # last period by default


In [ ]:
# any period, and a cross-section through the model
model.plot.map(layer=0, per=0).show()


### Package by package — inputs and results

In [ ]:
for pkg in ('chd', 'drn', 'ghb'):
    p = getattr(model.packages, pkg)
    print(f'--- {pkg} ---')
    print('  inputs nouns :', [a for a in dir(p.inputs) if not a.startswith('_')])
    print('  results nouns:', [a for a in dir(p.results) if not a.startswith('_')])


In [ ]:
model.packages.drn.inputs.summary()


In [ ]:
model.packages.drn.results.summary()


In [ ]:
model.packages.drn.inputs.map().show()


In [ ]:
# simulated flow through the drains, by period
# `results.q` is a noun; the verbs are get / summary / map / section / mosaic / animate.
model.packages.drn.results.q.summary()


In [ ]:
show(model.packages.drn.results.q.map(), 'simulated drain flow (ft3/d)')


## 9 · Side by side

**The USG run only saved heads for stress periods 61-72.** Its output control writes
nothing before that, so those twelve periods are the *only* window in which the two models
can be compared at all — and the full MF6 model currently stops at period 2.

That makes `compare()` mostly a guard for now. It exists because getting this wrong is easy
and convincing: comparing MF6's last record against USG's last record, when the two are
different times, produced a completely fictional agreement earlier in this work.

Once §10 gets the full model running past period 61, this becomes the real check.


In [ ]:
def compare(ws, period=None, layer=0):
    """Compare MF6 heads with the USG run at the SAME stress period.

    Returns None (with an explanation) when the two runs share no period, rather
    than quietly lining up records that are different times.
    """
    hm = flopy.utils.HeadFile(str(Path(ws) / 'tentrails.hds'))
    mf6_pers = {k[1] for k in hm.get_kstpkper()}
    usg_pers = {k[1] for k in usg_kk}
    shared = sorted(mf6_pers & usg_pers)
    if not shared:
        print(f'No shared stress period, so no comparison is possible.\n'
              f'  MF6 has periods {min(mf6_pers)+1}-{max(mf6_pers)+1}\n'
              f'  USG saved periods {min(usg_pers)+1}-{max(usg_pers)+1} (its OC writes nothing earlier)\n'
              f'  -> run the full model far enough to reach period {min(usg_pers)+1}.')
        return None
    per = shared[-1] if period is None else period
    km = max(k for k in hm.get_kstpkper() if k[1] == per)
    M = np.squeeze(np.array(hm.get_data(kstpkper=km)))
    M = np.where(np.abs(M) > 1e29, np.nan, M)
    ku = max(k for k in usg_kk if k[1] == per)
    U = np.array(usg_hds.get_data(kstpkper=ku))
    U = np.where(np.isclose(U, 999.0), np.nan, U)
    ok = np.isfinite(U) & np.isfinite(M)
    d = (M - U)[ok]
    print(f'stress period {per+1}  ({ok.sum()} cells)')
    print(f'  mean {d.mean():+.3f}  median {np.median(d):+.3f}  '
          f'|d| p90 {np.percentile(np.abs(d), 90):.2f}  max {np.abs(d).max():.2f} ft')
    for L in range(U.shape[0]):
        m = ok[L]
        if m.sum():
            dl = (M[L] - U[L])[m]
            print(f'    layer {L+1}: mean {dl.mean():+7.3f}  |d| p90 {np.percentile(np.abs(dl), 90):7.3f}')
    return U, M, per

result = compare(run.workspace)


Expect real differences: this short run has no recharge, no ET and no CLN. It is a check
that the *grid and boundaries* line up, not that the water balance does.

In [ ]:
# Only meaningful once the two runs share a period.
if result is not None:
    U, M, per = result
    show(vor.plot.map(M[0] - U[0], colorscale='RdBu'),
         f'MF6 - USG, layer 1, period {per+1} (ft)')


## 10 · Why the full model stops

`include=` restricts the converted boundary packages, so packages go back one at a time.
**Established: `('chd','ghb','drn')` reaches normal termination.** These cells finish it.

In [ ]:
def how_far(ws, label=''):
    """Report how far a run actually got -- not merely whether it exited 0."""
    lst_path = Path(ws) / 'mfsim.lst'
    lst = lst_path.read_text(errors='replace') if lst_path.exists() else ''
    ok = 'Normal termination' in lst
    try:
        kk = flopy.utils.HeadFile(str(Path(ws) / 'tentrails.hds')).get_kstpkper()
        reached = kk[-1][1] + 1
    except Exception:
        reached = 0
    print(f'{label:<20} normal={ok!s:<5} reached period {reached}')
    if 'ERROR REPORT' in lst:
        for e in re.findall(r'^\s+\d+\.\s+(.+)$', lst[lst.find('ERROR REPORT'):], re.M)[:3]:
            print(f'    ! {e.strip()}')
    return ok, reached


def try_packages(include, nper=4, label=None):
    ws = WORK / ('bisect_' + ('all' if include is None else '_'.join(include)))
    shutil.rmtree(ws, ignore_errors=True)
    spec = truncate(usg.to_mf6('tentrails', crs=CRS, start_date_time=START,
                               fix_for_mf6=True, include=include), nper)
    spec.build_flopy(ws).sim.write_simulation(silent=True)
    subprocess.run([MF6], cwd=ws, capture_output=True, text=True)
    return how_far(ws, label or str(include))

try_packages(('chd', 'ghb', 'drn'), label='chd+ghb+drn')   # known good


In [ ]:
try_packages(('chd', 'ghb', 'drn', 'rch'), label='+ rch')


In [ ]:
try_packages(('chd', 'ghb', 'drn', 'evt'), label='+ evt')


In [ ]:
try_packages(None, label='everything')


### Where the solver blows up

With `print_option='ALL'` MODFLOW 6 names the cell carrying the largest head change on each
outer iteration — the fastest route from "did not converge" to a cell number.

In [ ]:
def solver_trace(include=None, nper=3):
    ws = WORK / 'trace'; shutil.rmtree(ws, ignore_errors=True)
    spec = truncate(usg.to_mf6('tentrails', crs=CRS, start_date_time=START,
                               fix_for_mf6=True, include=include), nper)
    pk = [q for q in spec.packages if q.name != 'ims']
    pk.append(api.ims(models=('tentrails',), complexity='COMPLEX', print_option='ALL',
                      outer_maximum=100, inner_maximum=300))
    SimulationSpec(spec.name, models=spec.models,
                   packages=tuple(pk)).build_flopy(ws).sim.write_simulation(silent=True)
    subprocess.run([MF6], cwd=ws, capture_output=True, text=True)
    lst = (ws / 'mfsim.lst').read_text(errors='replace')
    rows = re.findall(r'Model\s+(\d+)\s+(\d+)\s+([-\d.E+]+)\s+\S*?\(([\d,]+)\)', lst)
    df = pd.DataFrame(rows, columns=['outer', 'inner', 'max_change', 'cellid'])
    df[['outer', 'inner']] = df[['outer', 'inner']].astype(int)
    df['max_change'] = df['max_change'].astype(float)
    return df, lst

trace, listing = solver_trace()
trace.tail(15)


In [ ]:
# The cells MF6 keeps naming, and what is special about them.
if len(trace):
    t = trace.assign(lay=trace.cellid.str.split(',').str[0].astype(int) - 1,
                     cell=trace.cellid.str.split(',').str[-1].astype(int) - 1)
    on_cln = set(usg.cln.cells_touched(usg.ncpl).tolist())
    rows = []
    for (lay, cell), chg in (t.groupby(['lay', 'cell']).max_change.max()
                              .abs().sort_values(ascending=False).head(10).items()):
        rows.append(dict(layer=lay + 1, cell=cell + 1, max_change=chg,
                         area=float(areas.iloc[cell]), thickness=usg.thickness[lay, cell],
                         k=usg.k[lay, cell], sy=usg.sy[lay, cell],
                         rch_max=float(R.max(axis=0)[cell]), on_cln=cell in on_cln))
    display(pd.DataFrame(rows).round(4))
else:
    print('no outer-iteration rows parsed -- check the regex against mfsim.lst')


In [8]:
model.plot.map(layer=-1, per=3, show_layer_elevs=True, show_mounding=True).show()

In [14]:
model.gwf.output.head().file

<_io.BufferedReader name='/tmp/usg_workbook/project/runs/short/tentrails.hds'>

### Test the recharge hypothesis directly

If the 2 ft/d cells are the cause, capping recharge should restore convergence. This edits
the *converted spec*, so the reader and the USG files are untouched.

In [ ]:
def try_capped_recharge(cap=0.05, nper=4):
    ws = WORK / f'cap_{cap}'; shutil.rmtree(ws, ignore_errors=True)
    spec = truncate(usg.to_mf6('tentrails', crs=CRS, start_date_time=START,
                               fix_for_mf6=True), nper)
    pkgs = []
    for p in spec.models[0].packages:
        if p.name == 'rcha':
            p = p.with_options(recharge={k: np.minimum(v, cap)
                                         for k, v in p.options['recharge'].items()})
        pkgs.append(p)
    m = ModelSpec('tentrails', 'gwf', packages=tuple(pkgs),
                  context=spec.models[0].context, options=spec.models[0].options)
    SimulationSpec(spec.name, models=(m,),
                   packages=spec.packages).build_flopy(ws).sim.write_simulation(silent=True)
    subprocess.run([MF6], cwd=ws, capture_output=True, text=True)
    return how_far(ws, f'recharge capped at {cap} ft/d')

try_capped_recharge(0.05)


## 11 · Taking it to a new grid

These three carry **no grid**, which is what makes them usable on the new mesh with the
infiltration-pond refinement.

- `surfaces(interpolate=True)` — layer contacts as `Surface` objects built from cell
  centres, so they evaluate anywhere. *(Built and returned; round-trip not yet verified.)*
- `boundary_frame(ftype)` — a boundary condition as a GeoDataFrame in real coordinates.
- `cln_polygons()` — the surface-water features, ready to become `LAK`/`SFR`.

In [ ]:
surfaces = usg.surfaces(interpolate=True)
print(f'{len(surfaces)} surfaces (top + {usg.nlay} bottoms)')
print(f'GHB frame: {len(ghb)} rows, crs {ghb.crs}')
print(f'CLN polygons: {len(clnp)} features, {clnp.geometry.area.sum():,.0f} ft2 total')


---

### The open questions this workbook exists to answer

1. **RCH, EVT, or both?** §10's bisect settles it.
2. **Is it the concentrated recharge specifically?** `try_capped_recharge()` tests it directly.
3. **Is the missing CLN the real reason?** Those water bodies received that recharge.
   Standing them back up as `LAK` — not crude CHD, which was tried and errored on sub-bottom
   heads — is the honest test, and the next real piece of work.
4. **Do the thin layer-1 cells matter?** Convertible cells 0.1 ft thick are classic Newton
   trouble, and §3 shows how many there are.
